# 00 · Auditoría de datos y piloto de accesibilidad estocástica

**Objetivo.** Verificar que el proyecto puede ejecutarse con datos públicos, descargar instantáneas reproducibles y construir un primer piloto Bogotá–Medellín.

El notebook no usa tarjetas inteligentes, telefonía, Waze ni credenciales institucionales.

In [ ]:
%pip -q install osmnx geopandas pyarrow pyyaml beautifulsoup4

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse
import hashlib, json, re, zipfile
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from bs4 import BeautifulSoup
from scipy import sparse
from scipy.sparse.linalg import spsolve
ROOT = Path('/content/colombia-stochastic-access')
RAW = ROOT / 'data/raw'; OUT = ROOT / 'outputs'
RAW.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
print('Workspace:', ROOT)

## 1. Registro de fuentes

Las fuentes obligatorias sostienen el análisis principal. Las opcionales son extensiones.

In [ ]:
sources = pd.DataFrame([
 {'id':'bogota_gtfs','city':'Bogotá','required':True,'kind':'GTFS','url':'https://files.mobilitydatabase.org/mdb-3358/mdb-3358-202607130343/mdb-3358-202607130343.zip'},
 {'id':'medellin_eod_trips','city':'Medellín','required':True,'kind':'EOD-viajes','url':'https://datosabiertos.metropol.gov.co/dataset/encuesta-origen-destino-2017-datos-por-viajes','direct_url':'https://datosabiertos.metropol.gov.co/sites/default/files/uploaded_resources/EOD_2017_DatosViajes_1_0.csv'},
 {'id':'medellin_eod_people','city':'Medellín','required':True,'kind':'EOD-moradores','url':'https://datosabiertos.metropol.gov.co/dataset/encuesta-origen-destino-2017-datos-por-moradores','direct_url':'https://datosabiertos.metropol.gov.co/sites/default/files/uploaded_resources/EOD_2017_DatosMoradores_0.csv'},
 {'id':'medellin_eod_households','city':'Medellín','required':True,'kind':'EOD-hogares','url':'https://datosabiertos.metropol.gov.co/dataset/encuesta-origen-destino-2017-datos-por-hogares','direct_url':'https://datosabiertos.metropol.gov.co/sites/default/files/uploaded_resources/EOD_2017_DatosHogares.csv'},
 {'id':'bogota_crashes','city':'Bogotá','required':False,'kind':'siniestros','url':'https://datosabiertos.bogota.gov.co/dataset/historico-siniestros-bogota-d-c'},
 {'id':'amva_open_data','city':'Medellín','required':False,'kind':'aire/SIATA','url':'https://datosabiertos.metropol.gov.co/'},
])
sources

In [ ]:
def probe(url, timeout=45):
 try:
  r=requests.get(url,stream=True,timeout=timeout,allow_redirects=True)
  return {'status':r.status_code,'content_type':r.headers.get('content-type',''),'bytes_header':r.headers.get('content-length'),'final_url':r.url}
 except Exception as exc: return {'status':'ERROR','content_type':'','bytes_header':None,'final_url':str(exc)}
audit=pd.concat([sources,sources.url.apply(probe).apply(pd.Series)],axis=1)
audit.to_csv(OUT/'source_audit.csv',index=False)
audit

## 2. Descarga reproducible del GTFS de Bogotá

Ocupa cerca de 115 MB. Cambie DOWNLOAD_GTFS a False para ejecutar solo la auditoría.

In [ ]:
DOWNLOAD_GTFS=True
def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for block in iter(lambda:f.read(1024**2),b''): h.update(block)
 return h.hexdigest()
def download(url,path,timeout=300):
 path=Path(path)
 if not path.exists() or path.stat().st_size == 0:
  with requests.get(url,stream=True,timeout=timeout) as r:
   r.raise_for_status(); total=int(r.headers.get('content-length',0)); done=0
   with path.open('wb') as f:
    for block in r.iter_content(1024**2):
     if block:
      f.write(block); done+=len(block)
      if total: print(f'\r{100*done/total:5.1f}%',end='')
  print()
 return {'url':url,'path':str(path),'bytes':path.stat().st_size,'sha256':sha256(path),'retrieved_utc':datetime.now(timezone.utc).isoformat()}
manifest=[]
gtfs_core_valid=False
gtfs_url=sources.loc[sources.id.eq('bogota_gtfs'),'url'].iloc[0]
gtfs_zip=RAW/'bogota_gtfs_2026-07-13.zip'
if DOWNLOAD_GTFS:
 manifest.append(download(gtfs_url,gtfs_zip))
 with zipfile.ZipFile(gtfs_zip) as zf:
  names=set(zf.namelist()); required={'agency.txt','routes.txt','stops.txt','trips.txt','stop_times.txt'}
  assert required.issubset(names), required-names
  routes=pd.read_csv(zf.open('routes.txt')); stops=pd.read_csv(zf.open('stops.txt'))
  gtfs_core_valid=True
 display(pd.Series({'routes':len(routes),'stops':len(stops),'valid_core':gtfs_core_valid}))

## 3. Descubrimiento, descarga y validación de la EOD de Medellín

El portal AMVA incrusta sus enlaces en metadatos DCAT/JSON. Se leen esos metadatos y, como respaldo reproducible, se conservan las URL oficiales directas de las tres tablas.

In [ ]:
suffixes=('.csv','.zip','.xlsx','.xls','.txt','.json','.geojson')
def is_machine_readable(url):
 clean=url.lower().split('?',1)[0]
 return clean.endswith(suffixes) or '/download/' in clean
def walk_metadata(value):
 if isinstance(value,dict):
  for key,item in value.items():
   if key.lower() in {'downloadurl','contenturl'} and isinstance(item,str): yield item
   yield from walk_metadata(item)
 elif isinstance(value,list):
  for item in value: yield from walk_metadata(item)
def decode_json_string(value):
 try: return json.loads(f'"{value}"')
 except (json.JSONDecodeError,TypeError): return value.replace('\\/','/')
def discover_files(catalog_url):
 r=requests.get(catalog_url,timeout=60); r.raise_for_status()
 soup=BeautifulSoup(r.text,'html.parser'); candidates=[]
 candidates.extend(a.get('href','') for a in soup.select('a[href]'))
 for script in soup.select('script[type="application/ld+json"], script[type="application/json"]'):
  payload=script.string or script.get_text()
  try: candidates.extend(walk_metadata(json.loads(payload)))
  except (json.JSONDecodeError,TypeError): pass
 for raw in re.findall(r'"(?:downloadURL|contentUrl)"\s*:\s*"([^"]+)"',r.text,flags=re.I):
  candidates.append(decode_json_string(raw))
 found=[]
 for candidate in candidates:
  if isinstance(candidate,str) and candidate:
   url=urljoin(catalog_url,candidate)
   if is_machine_readable(url): found.append(url)
 return list(dict.fromkeys(found))
eod_rows=[]
for row in sources[sources.kind.str.startswith('EOD')].itertuples():
 try:
  urls=discover_files(row.url)
  if pd.notna(row.direct_url): urls=[row.direct_url]+urls
  eod_rows.extend({'source_id':row.id,'resource_url':u} for u in dict.fromkeys(urls))
 except Exception as exc:
  if pd.notna(row.direct_url):
   eod_rows.append({'source_id':row.id,'resource_url':row.direct_url})
  else:
   eod_rows.append({'source_id':row.id,'resource_url':f'ERROR: {exc}'})
eod_resources=pd.DataFrame(eod_rows,columns=['source_id','resource_url'])
eod_resources.to_csv(OUT/'amva_eod_discovered_resources.csv',index=False)
eod_resources

In [ ]:
DOWNLOAD_EOD=True
def choose_resource(urls):
 preference={'.csv':0,'.zip':1,'.xlsx':2,'.xls':3,'.txt':4}
 candidates=[u for u in dict.fromkeys(urls) if is_machine_readable(u)]
 return min(candidates,key=lambda u:(preference.get(Path(urlparse(u).path).suffix.lower(),99),u.lower())) if candidates else None
def read_table_preview(path,nrows=5):
 suffix=path.suffix.lower()
 if suffix in {'.xlsx','.xls'}:
  frame=pd.read_excel(path,nrows=nrows)
 elif suffix in {'.csv','.txt'}:
  last_error=None
  for encoding in ('utf-8','latin-1'):
   try:
    frame=pd.read_csv(path,sep=None,engine='python',nrows=nrows,encoding=encoding)
    break
   except Exception as exc: last_error=exc
  else: raise last_error
 else:
  raise ValueError(f'Formato tabular no soportado para validación: {suffix}')
 if frame.shape[1] == 0: raise ValueError('La tabla no contiene columnas')
 return frame
eod_checks=[]
required_eod=sources.loc[sources.kind.str.startswith('EOD') & sources.required,'id'].tolist()
if DOWNLOAD_EOD:
 for source_id in required_eod:
  group=eod_resources[eod_resources.source_id.eq(source_id)]
  urls=[u for u in group.resource_url if not u.startswith('ERROR:')]
  url=choose_resource(urls)
  check={'source_id':source_id,'resource_url':url,'downloaded':False,'readable':False,'bytes':0,'preview_rows':0,'columns':0,'error':None}
  if url is None:
   check['error']='No se encontró recurso tabular'
   eod_checks.append(check); continue
  suffix=Path(urlparse(url).path).suffix.lower() or '.dat'
  path=RAW/f'{source_id}{suffix}'
  try:
   record=download(url,path); manifest.append(record)
   preview=read_table_preview(path)
   check.update(downloaded=path.exists() and path.stat().st_size>0,readable=True,bytes=path.stat().st_size,preview_rows=len(preview),columns=len(preview.columns))
  except Exception as exc:
   check['error']=f'{type(exc).__name__}: {exc}'
  eod_checks.append(check)
eod_validation=pd.DataFrame(eod_checks)
(RAW/'manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
display(eod_validation)
pd.DataFrame(manifest)

## 4. Redes peatonales piloto desde OpenStreetMap

Radios pequeños comprueban el flujo. El análisis final se agregará por zona censal o de transporte.

In [ ]:
pilots={'Bogotá–Chapinero':(4.6280,-74.0640),'Medellín–San Javier':(6.2567,-75.6142)}
WALK_SPEED_KPH=4.5
graphs={}
fig,axes=plt.subplots(1,2,figsize=(13,6))
for ax,(name,point) in zip(axes,pilots.items()):
 G=ox.graph_from_point(point,dist=1200,network_type='walk',simplify=True)
 for _,_,_,data in G.edges(keys=True,data=True):
  data['speed_kph']=WALK_SPEED_KPH
  data['travel_time']=float(data['length'])/(WALK_SPEED_KPH/3.6)
 graphs[name]=G
 ox.plot_graph(G,ax=ax,node_size=0,edge_linewidth=.6,edge_color='#3346b8',bgcolor='white',show=False,close=False)
 ax.set_title(f'{name}\n{len(G):,} nodos · {G.number_of_edges():,} aristas')
plt.tight_layout(); plt.savefig(OUT/'pilot_networks.png',dpi=180,bbox_inches='tight')

## 5. Prueba matemática de primer paso

Valida la maquinaria en una red pequeña antes de incorporar servicios esenciales.

In [ ]:
def transition_matrix(adjacency,costs=None,beta=1.0):
 A=sparse.csr_matrix(adjacency,dtype=float)
 if costs is None: W=A.copy()
 else:
  C=sparse.csr_matrix(costs,dtype=float); rows,cols=A.nonzero()
  W=sparse.csr_matrix((np.exp(-beta*np.asarray(C[rows,cols]).ravel()),(rows,cols)),shape=A.shape)
 sums=np.asarray(W.sum(axis=1)).ravel(); isolated=np.flatnonzero(sums==0)
 if len(isolated):
  W+=sparse.csr_matrix((np.ones(len(isolated)),(isolated,isolated)),shape=A.shape); sums=np.asarray(W.sum(axis=1)).ravel()
 return sparse.diags(1/sums)@W
def mean_first_passage(P,targets):
 n=P.shape[0]; transient=np.setdiff1d(np.arange(n),np.asarray(targets)); Q=P[transient][:,transient]; m=np.zeros(n)
 m[transient]=spsolve(sparse.eye(len(transient))-Q,np.ones(len(transient))); return m
A=np.array([[0,1,1,0],[0,0,1,1],[0,0,0,1],[0,0,0,1]],float)
C=np.array([[0,1,3,0],[0,0,1,2],[0,0,0,1],[0,0,0,0]],float)
P=transition_matrix(A,C,beta=.8); mfpt=mean_first_passage(P,[3])
pd.DataFrame({'node':range(4),'MFPT_steps':mfpt})

## 6. Compuerta básica de factibilidad

El piloto pasa únicamente si: (1) el GTFS de Bogotá contiene sus tablas núcleo, (2) las tres tablas EOD de Medellín fueron descargadas y son legibles, y (3) ambas redes OSM tienen nodos y aristas.

In [ ]:
valid_eod=set()
if not eod_validation.empty:
 valid_mask=eod_validation.downloaded.fillna(False) & eod_validation.readable.fillna(False)
 valid_eod=set(eod_validation.loc[valid_mask,'source_id'])
gate={'bogota_gtfs_core':bool(gtfs_core_valid),
      'medellin_eod_three_tables':set(required_eod).issubset(valid_eod),
      'osm_two_city_pilot':len(graphs)==2 and all(len(G)>0 and G.number_of_edges()>0 for G in graphs.values())}
gate['gate_1_passed']=all(gate.values())
pd.Series(gate,name='result')